> ## SOLUTION / ANSWER KEY &mdash; Lab 5.8
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-08-guardrails.ipynb`](../lab-08-guardrails.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 5.8 &mdash; Guardrails: Keep the Agent Safe

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Allow-list the tools an agent may call
- Detect a runaway loop (the same action repeated)
- Validate tool input and stop a runaway agent safely

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 5 slides &mdash; Guardrails &amp; human-in-the-loop](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-05-08"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Autonomy needs **guardrails**. Four cheap, essential ones: a **max_steps** cap (already in the
loop), a **tool allow-list** (only call permitted tools), **loop detection** (stop if the same
action repeats), and **input validation** (reject dangerous tool inputs). Together they stop an
agent that hallucinates a tool, gets stuck, or is fed garbage.

In [ ]:
# DEMO
ALLOWED = {"lookup", "calculator"}
ALLOWED_CHARS = set("0123456789+-*/(). ")
print("allowed tools:", ALLOWED)
print("'rm -rf /' allowed as a tool?", "rm_rf" in ALLOWED)

## Your Turn
Implement the three guards, then see them stop a deliberately runaway agent.

In [ ]:
ALLOWED = {"lookup", "calculator"}
ALLOWED_CHARS = set("0123456789+-*/(). ")

def is_allowed(action):
    return action in ALLOWED

def detect_loop(memory, k=3):
    actions = [s["action"] for s in memory]
    return len(actions) >= k and len(set(actions[-k:])) == 1

def is_valid_calc_input(expr):
    return all(c in ALLOWED_CHARS for c in expr) and any(c.isdigit() for c in expr)

# A guarded runner that uses the guards above (provided)
def safe_run(policy, max_steps=10):
    memory = []
    for i in range(max_steps):
        action, arg = policy(memory)
        if not is_allowed(action):
            return {'stopped': 'blocked_tool', 'steps': i}
        if detect_loop(memory + [{'action': action}]):
            return {'stopped': 'loop_detected', 'steps': i}
        memory.append({'action': action, 'observation': '...'})
    return {'stopped': 'max_steps', 'steps': max_steps}

try:
    print('runaway (always lookup):', safe_run(lambda m: ('lookup', 'x')))
    print('bad tool (rm_rf):       ', safe_run(lambda m: ('rm_rf', '/')))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("allow-list permits a known tool", lambda: is_allowed("calculator") is True)
expect_true("allow-list blocks an unknown tool", lambda: is_allowed("delete_database") is False)
expect_true("loop detector fires on 3 repeats", lambda: detect_loop([{"action": "lookup"}] * 3) is True)
expect_true("loop detector tolerates varied actions", lambda: detect_loop([{"action": "lookup"}, {"action": "calculator"}, {"action": "lookup"}]) is False)
expect_true("valid calc input accepted", lambda: is_valid_calc_input("2 + 2*3") is True)
expect_true("dangerous calc input rejected", lambda: is_valid_calc_input("__import__('os')") is False)
expect_true("a runaway agent is stopped, not infinite", lambda: safe_run(lambda m: ("lookup", "x"))["stopped"] in {"loop_detected", "max_steps"})
expect_true("a disallowed tool is blocked at step 0", lambda: safe_run(lambda m: ("rm_rf", "/"))["stopped"] == "blocked_tool")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Allow-list, loop detection and input validation turn an autonomous agent from a liability into a tool you can trust. Day 5 goes deeper on responsible agents.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>